In [7]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"
with open(redlist_file, "r") as f:
    num_rows = sum(1 for line in f if line.strip())

print(num_rows)

# ============================================================
# Load catalogs
# ============================================================
colnames = ["ID", "X", "Y", "RA", "DEC",
            "MAG_APER", "MAGERR_APER",
            "MAG_AUTO", "MAGERR_AUTO", "FLAGS"]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.1564, 0.0039)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
"""def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")"""
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)
    df["MAGERR_AUTO_CAL"] = np.sqrt(df["MAGERR_AUTO"]**2 + zp_err**2)
calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")    

# ============================================================
# SNR function
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0  # arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matching helper
# ============================================================
def matched_array(src_df, idx, sep, limit_mag, err_col="MAGERR_CAL"):
    mag = np.full(len(y_df), limit_mag)
    err = np.full(len(y_df), 9.9)

    ok = sep.arcsec < match_radius
    idx = np.asarray(idx).astype(int)

    mag[ok] = src_df.MAG_CAL.values[idx[ok]]
    err[ok] = src_df[err_col].values[idx[ok]]

    return mag, err


# Limiting magnitudes
#i_mag, i_err = matched_array(i_df, idx_i, sep_i, 27.38)
i_mag, i_err_aper = matched_array(
    i_df, idx_i, sep_i, 27.38, err_col="MAGERR_CAL"
)

_, i_err_kron = matched_array(
    i_df, idx_i, sep_i, 27.38, err_col="MAGERR_AUTO_CAL"
)


z_mag, z_err = matched_array(z_df, idx_z, sep_z, 27.32)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# SNR
# ============================================================
i_snr_aper = magerr_to_snr(i_err_aper)
i_snr_kron = magerr_to_snr(i_err_kron)


# ============================================================
# LBG COLOR + DROPOUT CUTS
# ============================================================
cut_zy      = color_zy > 0.8
cut_break   = color_iz > 1.0
cut_faint_i = (i_snr_aper < 2.0) & (i_snr_kron < 2.0)
cut_sig     = np.abs(color_zy) > 2.5 * color_zy_err

      # robust Z detection






# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_zy &
    cut_break &
    cut_faint_i &
    cut_sig 
)

lbg_idx = np.where(final_sel)[0]
print(f"After color + quality cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_SNR_APER": i_snr_aper[lbg_idx],
    "I_SNR_KRON": i_snr_kron[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")
lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal detected sources (RA, DEC, I_SNR_APER, I_SNR_KRON):")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA = {row.RA:.6f}, "
        f"DEC = {row.DEC:.6f}, "
        f"I_SNR_APER = {row.I_SNR_APER:.2f}, "
        f"I_SNR_KRON = {row.I_SNR_KRON:.2f}"
    )


1148
Y catalog size: 408754
After color + quality cuts: 900 candidates
Final LBG candidates after red-list removal: 899
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final detected sources (RA, DEC, I_SNR_APER, I_SNR_KRON):
RA = 357.319567, DEC = -31.803504, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.519923, DEC = -31.800402, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.364103, DEC = -31.798601, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.452877, DEC = -31.794246, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.540868, DEC = -31.793045, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.388674, DEC = -31.793125, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 356.684894, DEC = -31.773376, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.312668, DEC = -31.748975, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 356.746755, DEC = -31.740070, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.311106, DEC = -31.706540, I_SNR_APER = 0.11, I_SNR_KRON = 0.11
RA = 357.314675, DEC = -31.7000

In [ ]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = ["ID", "X", "Y", "RA", "DEC",
            "MAG_APER", "MAGERR_APER",
            "MAG_AUTO", "MAGERR_AUTO", "FLAGS"]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.1564, 0.0039)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_APER_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_APER_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)
    df["MAG_AUTO_CAL"] = df["MAG_AUTO"] + zp
    df["MAGERR_AUTO_CAL"] = np.sqrt(df["MAGERR_AUTO"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR function
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0  # arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matching helper
# ============================================================
def matched_array(mag, err, idx, sep, limit_mag=99.):
    mag_out = np.full(len(y_df), limit_mag)
    err_out = np.full(len(y_df), 9.9)
    ok = sep.arcsec < match_radius
    mag_out[ok] = mag.values[idx[ok]]
    err_out[ok] = err.values[idx[ok]]
    return mag_out, err_out

# ============================================================
# Build matched photometry
# ============================================================
i_mag_aper, i_err_aper = matched_array(
    i_df.MAG_APER_CAL, i_df.MAGERR_APER_CAL, idx_i, sep_i
)

i_mag_auto, i_err_auto = matched_array(
    i_df.MAG_AUTO_CAL, i_df.MAGERR_AUTO_CAL, idx_i, sep_i
)

z_mag, z_err = matched_array(
    z_df.MAG_APER_CAL, z_df.MAGERR_APER_CAL, idx_z, sep_z
)

y_mag = y_df.MAG_APER_CAL.values
y_err = y_df.MAGERR_APER_CAL.values

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag_aper - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# SNRs
# ============================================================
i_snr_aper = magerr_to_snr(i_err_aper)
i_snr_auto = magerr_to_snr(i_err_auto)
y_snr = magerr_to_snr(y_err)

# ============================================================
# LBG COLOR + DROPOUT CUTS
# ============================================================
cut_zy        = color_zy > 0.8
cut_break    = color_iz > 1.0

# I-band must be non-detected in BOTH aperture and Kron
cut_i_faint  = (i_snr_aper < 1.0) & (i_snr_auto < 1.0)

# Y-band must be detected
cut_y_detect = y_snr >= 5.0

# Color significance
cut_sig = np.abs(color_zy) > 2.5 * color_zy_err
color_iz_err = np.sqrt(i_err_aper**2 + z_err**2)
cut_iz_sig   = color_iz > 2.5 * color_iz_err


# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_zy &
    cut_break &
    cut_i_faint &
    cut_y_detect &
    cut_sig &
    cut_iz_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After LBG selection: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "I_mag_aper": i_mag_aper[lbg_idx],
    "I_mag_auto": i_mag_auto[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_SNR_APER": i_snr_aper[lbg_idx],
    "I_SNR_AUTO": i_snr_auto[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")
lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")


Y catalog size: 408754
After LBG selection: 799 candidates
Final LBG candidates after red-list removal: 799
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat


In [53]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = ["ID", "X", "Y", "RA", "DEC",
            "MAG_APER", "MAGERR_APER",
            "MAG_AUTO", "MAGERR_AUTO", "FLAGS"]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.1564, 0.0039)
}

# ============================================================
# Limiting magnitudes (IMPORTANT)
# ============================================================
I_LIM = 27.32
Z_LIM = 27.38

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_APER_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_APER_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)
    df["MAG_AUTO_CAL"] = df["MAG_AUTO"] + zp
    df["MAGERR_AUTO_CAL"] = np.sqrt(df["MAGERR_AUTO"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR function
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0  # arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matching helper
# ============================================================
def matched_array(mag, err, idx, sep, limit_mag=99.):
    mag_out = np.full(len(y_df), limit_mag)
    err_out = np.full(len(y_df), 9.9)
    ok = sep.arcsec < match_radius
    mag_out[ok] = mag.values[idx[ok]]
    err_out[ok] = err.values[idx[ok]]
    return mag_out, err_out

# ============================================================
# Build matched photometry
# ============================================================
i_mag_aper, i_err_aper = matched_array(
    i_df.MAG_APER_CAL, i_df.MAGERR_APER_CAL, idx_i, sep_i
)

i_mag_auto, i_err_auto = matched_array(
    i_df.MAG_AUTO_CAL, i_df.MAGERR_AUTO_CAL, idx_i, sep_i
)

z_mag, z_err = matched_array(
    z_df.MAG_APER_CAL, z_df.MAGERR_APER_CAL, idx_z, sep_z
)

y_mag = y_df.MAG_APER_CAL.values
y_err = y_df.MAGERR_APER_CAL.values

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag_aper - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)
color_iz_err = np.sqrt(i_err_aper**2 + z_err**2)

# ============================================================
# SNRs
# ============================================================
i_snr_aper = magerr_to_snr(i_err_aper)
i_snr_auto = magerr_to_snr(i_err_auto)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# LBG SELECTION CUTS (PHYSICALLY CORRECT)
# ============================================================

# Z–Y red continuum
cut_zy = color_zy > 0.8
cut_zy_upper = color_zy < 2.5   # remove dusty low-z

# Strong Lyman break
cut_break = color_iz > 1.0
cut_iz_sig = color_iz > 2.5 * color_iz_err

# I-band dropout (SNR OR magnitude based)
cut_i_nondet = (
    ((i_snr_aper < 1.0) & (i_snr_auto < 1.0)) 
)



# Y-band must be detected (NO limiting mag)
cut_y_detect = y_snr >= 5.0

# Color significance
cut_zy_sig = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_zy &
    cut_zy_upper &
    cut_break &
    cut_i_nondet &
    cut_y_detect &
    cut_zy_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After LBG selection: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "I_mag_aper": i_mag_aper[lbg_idx],
    "I_mag_auto": i_mag_auto[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_SNR_APER": i_snr_aper[lbg_idx],
    "I_SNR_AUTO": i_snr_auto[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")
lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")


Y catalog size: 408754
After LBG selection: 548 candidates
Final LBG candidates after red-list removal: 548
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat


In [42]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = ["ID", "X", "Y", "RA", "DEC",
            "MAG_APER", "MAGERR_APER",
            "MAG_AUTO", "MAGERR_AUTO", "FLAGS"]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.1564, 0.0039)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR (ONLY for real detections)
# ============================================================
def magerr_to_snr(err):
    return 2.5 / (np.log(10) * err)

# ============================================================
# Coordinate matching (Y as reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0  # arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Proper matching: NO fake mags, NO fake errors
# ============================================================
def matched_photometry(src_df, idx, sep):
    mag = np.full(len(y_df), np.nan)
    err = np.full(len(y_df), np.nan)
    detected = sep.arcsec < match_radius
    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]
    return mag, err, detected

i_mag, i_err, i_det = matched_photometry(i_df, idx_i, sep_i)
z_mag, z_err, z_det = matched_photometry(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# Colors (only where defined)
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# I-band SNR (only if detected)
# ============================================================
i_snr = np.full(len(y_df), np.nan)
i_snr[i_det] = magerr_to_snr(i_err[i_det])

# ============================================================
# LBG / DROPOUT SELECTION
# ============================================================

# Strong z–y break
cut_zy = color_zy > 0.8

# i-dropout condition:
#   EITHER undetected in I
#   OR detected but very low SNR
cut_i_dropout = (~i_det) | (i_snr < 2.0)

# Remove catastrophic fake colors
cut_sig = np.abs(color_zy) > 2.5 * color_zy_err

# Quality cuts
cut_z_err = z_err < 0.9
cut_y_err = y_err < 0.9

# ============================================================
# Final selection
# ============================================================
final_sel = (
    z_det &                    # must be detected in Z
    cut_zy &
    cut_i_dropout &
    cut_sig &
    cut_z_err &
    cut_y_err
)

lbg_idx = np.where(final_sel)[0]
print(f"After selection: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")
lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal detected sources:")
for _, row in lbg_candidates.iterrows():
    print(f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, I_SNR={row.I_SNR}")


Y catalog size: 518449
After selection: 3722 candidates
Final LBG candidates: 3719
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final detected sources:
RA=357.319567, DEC=-31.803504, I_SNR=nan
RA=357.463867, DEC=-31.802417, I_SNR=nan
RA=357.338180, DEC=-31.802908, I_SNR=nan
RA=357.338457, DEC=-31.803205, I_SNR=nan
RA=357.372225, DEC=-31.801037, I_SNR=nan
RA=357.430458, DEC=-31.798854, I_SNR=nan
RA=357.364134, DEC=-31.798587, I_SNR=nan
RA=357.452962, DEC=-31.794510, I_SNR=nan
RA=357.520486, DEC=-31.793945, I_SNR=nan
RA=357.315826, DEC=-31.794453, I_SNR=nan
RA=357.540167, DEC=-31.793528, I_SNR=nan
RA=357.540868, DEC=-31.793045, I_SNR=nan
RA=357.467557, DEC=-31.792790, I_SNR=nan
RA=357.388653, DEC=-31.793080, I_SNR=nan
RA=357.448259, DEC=-31.791813, I_SNR=nan
RA=357.527314, DEC=-31.791094, I_SNR=nan
RA=356.763950, DEC=-31.790447, I_SNR=nan
RA=356.728394, DEC=-31.789866, I_SNR=nan
RA=357.472336, DEC=-31.789516, I_SNR=nan
RA=356.800404, DEC=-31.788224, I_SNR=nan
RA

In [ ]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.1564, 0.0039)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)


def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)


color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)



cut_z_detected = z_detected
cut_z_snr      = z_snr > 3.0        # robust Z detection
cut_y_snr      = magerr_to_snr(y_err) > 3.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

cut_i_dropout  = (~i_detected) | (i_snr < 2.0)   # ONLY SNR logic
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After SNR-only cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f} "
        f"I_mag={row.I_mag:.2f}, z_mag={row.Z_mag:.2f}, y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 408754
After SNR-only cuts: 136 candidates
Final LBG candidates after red-list removal: 135
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final candidates:
RA=357.253316, DEC=-31.657425, I_SNR=0.01 I_mag=129.70, z_mag=25.69, y_mag=24.42
RA=357.299685, DEC=-31.657749, I_SNR=1.52 I_mag=26.79, z_mag=25.26, y_mag=24.40
RA=357.228167, DEC=-31.656915, I_SNR=0.26 I_mag=28.73, z_mag=25.80, y_mag=24.66
RA=357.239016, DEC=-31.656564, I_SNR=1.01 I_mag=27.24, z_mag=25.60, y_mag=24.59
RA=357.190553, DEC=-31.654978, I_SNR=0.31 I_mag=28.53, z_mag=25.68, y_mag=24.67
RA=357.430293, DEC=-31.651225, I_SNR=1.98 I_mag=26.51, z_mag=25.32, y_mag=24.52
RA=357.531209, DEC=-31.632462, I_SNR=0.79 I_mag=27.51, z_mag=25.60, y_mag=24.16
RA=357.184588, DEC=-31.587619, I_SNR=0.01 I_mag=129.70, z_mag=24.63, y_mag=23.78
RA=357.260188, DEC=-31.580653, I_SNR=0.71 I_mag=27.66, z_mag=25.53, y_mag=24.50
RA=357.156439, DEC=-31.538571, I_SNR=0.01 I_mag=129.70, z_mag=24.96, y_mag=24.03


In [32]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.8603, 0.0024),
    "z": (30.4519, 0.0036),
    "y": (28.2513, 0.0029)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)

# ============================================================
# FORCE I-band non-detections to limiting magnitude
# ============================================================
I_LIMIT = 27.46

i_nondet = (~i_detected) | (i_snr < 2.0)

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = np.inf
i_snr[i_nondet] = 0.0

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 3.0
cut_y_snr      = magerr_to_snr(y_err) > 3.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

cut_i_dropout  = i_nondet
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After SNR-only cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, z_mag={row.Z_mag:.2f}, y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 413426
After SNR-only cuts: 1814 candidates
Final LBG candidates after red-list removal: 1813
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final candidates:
RA=357.319150, DEC=-31.804170, I_SNR=0.00, I_mag=27.46, z_mag=25.01, y_mag=24.20
RA=357.319567, DEC=-31.803504, I_SNR=0.00, I_mag=27.46, z_mag=25.49, y_mag=24.35
RA=357.366107, DEC=-31.803211, I_SNR=0.00, I_mag=27.46, z_mag=25.29, y_mag=24.46
RA=357.364103, DEC=-31.798601, I_SNR=0.00, I_mag=27.46, z_mag=25.42, y_mag=24.43
RA=357.370676, DEC=-31.795678, I_SNR=0.00, I_mag=27.46, z_mag=25.95, y_mag=24.83
RA=357.467519, DEC=-31.792734, I_SNR=0.00, I_mag=27.46, z_mag=25.38, y_mag=24.48
RA=357.388674, DEC=-31.793125, I_SNR=0.00, I_mag=27.46, z_mag=25.95, y_mag=24.75
RA=356.777390, DEC=-31.791902, I_SNR=0.00, I_mag=27.46, z_mag=25.60, y_mag=24.68
RA=357.319171, DEC=-31.791321, I_SNR=0.00, I_mag=27.46, z_mag=24.76, y_mag=23.87
RA=356.800532, DEC=-31.788252, I_SNR=0.00, I_mag=27.46, z_mag=25.55, y_m

In [29]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.4754, 0.0029)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)

# ============================================================
# Define I-band non-detections (DO NOT modify values yet)
# ============================================================
I_LIMIT = 27.26
i_nondet = (~i_detected) | (i_snr < 2.0)

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 3.0
cut_y_snr      = magerr_to_snr(y_err) > 3.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

cut_i_dropout  = i_nondet
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Apply I-band upper limit ONLY where appropriate
# ============================================================
i_replace = i_nondet & cut_break

i_mag[i_replace] = I_LIMIT
i_err[i_replace] = np.inf   # upper limit, but SNR untouched

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After SNR-only cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, z_mag={row.Z_mag:.2f}, y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 408754
After SNR-only cuts: 59 candidates
Final LBG candidates after red-list removal: 59
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final candidates:
RA=357.253316, DEC=-31.657425, I_SNR=0.01, I_mag=27.26, z_mag=25.69, y_mag=24.73
RA=357.531209, DEC=-31.632462, I_SNR=0.79, I_mag=27.26, z_mag=25.60, y_mag=24.48
RA=356.736997, DEC=-31.484876, I_SNR=1.43, I_mag=27.26, z_mag=25.17, y_mag=24.33
RA=356.612813, DEC=-31.433612, I_SNR=0.65, I_mag=27.26, z_mag=25.70, y_mag=24.39
RA=357.415268, DEC=-31.408956, I_SNR=1.51, I_mag=27.26, z_mag=25.07, y_mag=24.21
RA=356.317534, DEC=-31.337865, I_SNR=0.03, I_mag=27.26, z_mag=25.49, y_mag=24.46
RA=357.311973, DEC=-31.339543, I_SNR=0.81, I_mag=27.26, z_mag=25.81, y_mag=24.73
RA=356.708144, DEC=-31.328626, I_SNR=0.71, I_mag=27.26, z_mag=25.85, y_mag=24.65
RA=356.189648, DEC=-31.315766, I_SNR=0.01, I_mag=27.26, z_mag=25.51, y_mag=24.28
RA=356.504840, DEC=-31.317335, I_SNR=1.72, I_mag=27.26, z_mag=25.42, y_mag=2

In [74]:
print(np.sqrt(0.59)*5)

3.8405728739343044


In [48]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list.txt"
with open(redlist_file, "r") as f:
    num_rows = sum(1 for line in f if line.strip())

print(num_rows)

# ============================================================
# Load catalogs
# ============================================================
colnames = ["ID", "X", "Y", "RA", "DEC",
            "MAG_APER", "MAGERR_APER",
            "MAG_AUTO", "MAGERR_AUTO", "FLAGS"]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size (raw): {len(y_df)}")

# ============================================================
# REMOVE bogus Y-band sources (MAG > 90) → propagates to all bands
# ============================================================
good_y = y_df["MAG_APER"] < 90
y_df = y_df[good_y].reset_index(drop=True)

print(f"Y catalog size after MAG>90 removal: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.700, 0.003),
    "z": (30.214, 0.007),
    "y": (28.1564, 0.0039)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR function
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0  # arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matching helper
# ============================================================
def matched_array(src_df, idx, sep, limit_mag):
    mag = np.full(len(y_df), limit_mag)
    err = np.full(len(y_df), 9.9)
    ok = sep.arcsec < match_radius
    mag[ok] = src_df.MAG_CAL.values[idx[ok]]
    err[ok] = src_df.MAGERR_CAL.values[idx[ok]]
    return mag, err

# Limiting magnitudes
i_mag, i_err = matched_array(i_df, idx_i, sep_i, 27.58)
z_mag, z_err = matched_array(z_df, idx_z, sep_z, 27.42)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)

# ============================================================
# LBG COLOR + DROPOUT CUTS
# ============================================================
cut_zy      = color_zy > 0.5
cut_break   = color_iz > 1.0
cut_faint_i = i_snr < 2.0
cut_sig     = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# QUALITY CUTS
# (keeps 0.3 cut, removes only extreme bogus errors like 9.9)
# ============================================================
cut_z_mag   = z_mag < 27.32
cut_y_mag   = y_mag < 26.79

cut_z_err   = z_err < 0.3
cut_y_err   = y_err < 0.3

cut_no_bad_err = (z_err < 9.0) & (y_err < 9.0)

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_zy &
    cut_break &
    cut_faint_i &
    cut_sig &
    cut_z_mag &
    cut_y_mag &
    cut_z_err &
    cut_y_err &
    cut_no_bad_err
)

lbg_idx = np.where(final_sel)[0]
print(f"After color + quality cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")
lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")


1148
Y catalog size (raw): 408754
Y catalog size after MAG>90 removal: 399006
After color + quality cuts: 903 candidates
Final LBG candidates after red-list removal: 902
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat


In [25]:
import pandas as pd
import os


def cat_to_reg(cat_file,
               ra_col="RA",
               dec_col="DEC",
               mag_cols=None,
               point_style="circle",
               point_size=6,
               color="cyan",
               radius_arcsec=1.0):
    """
    Convert a catalog to DS9 region files (points + 1 arcsec circles)
    with labels including RA, Dec, and magnitudes.

    Parameters
    ----------
    cat_file : str
        Path to input catalog
    ra_col, dec_col : str
        RA / Dec column names
    mag_cols : list
        List of magnitude column names to label
    point_style : str
        DS9 point style
    point_size : int
        Point size
    color : str
        DS9 region color
    radius_arcsec : float
        Circle radius in arcseconds
    """

    if mag_cols is None:
        mag_cols = []

    base = cat_file.replace(".cat", "")
    reg_point = base + "_points.reg"
    reg_circle = base + "_r1arcsec.reg"

    df = pd.read_csv(cat_file)

    ra = df[ra_col].values
    dec = df[dec_col].values

    # Build label text
    """def make_label(row):
        label = f"RA={row[ra_col]:.6f}, Dec={row[dec_col]:.6f}"
        for m in mag_cols:
            if m in row:
                label += f", {m}={row[m]:.2f}"
        return label"""

    # ---------- POINT REGIONS ----------
    with open(reg_point, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        """for _, row in df.iterrows():
            #label = make_label(row)
            f.write(
                f"point({row[ra_col]},{row[dec_col]}) "
                f"# point={point_style} {point_size} text={{{label}}}\n"
            )"""

    # ---------- 1 ARCSEC CIRCLES ----------
    with open(reg_circle, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        """for _, row in df.iterrows():
            label = make_label(row)
            f.write(
                f'circle({row[ra_col]},{row[dec_col]},{radius_arcsec}") '
                f"# text={{{label}}}\n"
            )"""

    print(f"Created:")
    print(f"  {reg_point}")
    print(f"  {reg_circle}")


if __name__ == "__main__":

    CAT_FILES = [
        "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
    ]

    for cat in CAT_FILES:
        if not os.path.exists(cat):
            print(f"File not found: {cat}")
            continue

        cat_to_reg(
            cat_file=cat,
            ra_col="RA",
            dec_col="DEC",
            mag_cols=["MAG_I", "MAG_Z", "MAG_Y"],  # <-- EDIT THIS
            point_style="circle",
            point_size=6,
            color="cyan",
            radius_arcsec=1.0
        )

    print("All catalogs converted to DS9 region files.")


Created:
  /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY_points.reg
  /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY_r1arcsec.reg
All catalogs converted to DS9 region files.


In [33]:
import pandas as pd
import os


def cat_to_reg(cat_file,
               ra_col="RA",
               dec_col="DEC",
               point_style="circle",
               point_size=6,
               color="cyan",
               radius_arcsec=1.0):
    """
    Convert a catalog to DS9 region files:
    - point markers
    - 1 arcsec radius circles
    (NO labels)
    """

    base = cat_file.replace(".cat", "")
    reg_point = base + "_points.reg"
    reg_circle = base + "_r1arcsec.reg"

    df = pd.read_csv(cat_file)

    # ---------- POINT REGIONS ----------
    with open(reg_point, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        for _, row in df.iterrows():
            f.write(
                f"point({row[ra_col]},{row[dec_col]}) "
                f"# point={point_style} {point_size}\n"
            )

    # ---------- 1 ARCSEC CIRCLES ----------
    with open(reg_circle, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        for _, row in df.iterrows():
            f.write(
                f'circle({row[ra_col]},{row[dec_col]},{radius_arcsec}")\n'
            )

    print(f"Created:")
    print(f"  {reg_point}")
    print(f"  {reg_circle}")


if __name__ == "__main__":

    CAT_FILES = [
        "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/cat_depth/matched_decam_ydepth_new.cat"
    ]

    for cat in CAT_FILES:
        if not os.path.exists(cat):
            print(f"File not found: {cat}")
            continue

        cat_to_reg(
            cat_file=cat,
            ra_col="RA",
            dec_col="DEC",
            point_style="circle",
            point_size=6,
            color="cyan",
            radius_arcsec=1.0
        )

    print("All catalogs converted to DS9 region files.")


KeyError: 'RA'

In [23]:
import os
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from regions import Regions

# -------------------------
# Paths
# -------------------------
region_file = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.reg"
fits_image = "/Users/aishwarya/Desktop/Lyman_alpha_2/Y_band/Trim/trim2deg_mosaic_y.fits"
output_dir = "/Users/aishwarya/Desktop/Lyman_alpha_2/Snapshot_LBGs"

# -------------------------
# Clean output directory
# -------------------------
os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    if os.path.isfile(file_path):
        os.remove(file_path)

print("  Existing snapshot files deleted.")

# -------------------------
# Cutout size (pixels)
# -------------------------
cutout_size = (50, 50)

# -------------------------
# Load FITS image
# -------------------------
with fits.open(fits_image) as hdul:
    data = hdul[0].data
    header = hdul[0].header
    wcs = WCS(header)

# -------------------------
# Load regions
# -------------------------
regions = Regions.read(region_file, format="ds9")

# -------------------------
# Loop over regions
# -------------------------
for i, reg in enumerate(regions, start=1):

    # Sky coordinates
    sky_center = reg.center
    ra = sky_center.ra.deg
    dec = sky_center.dec.deg

    # Convert to pixel coordinates
    pixel_center = wcs.world_to_pixel(sky_center)

    try:
        cutout = Cutout2D(
            data,
            position=pixel_center,
            size=cutout_size,
            wcs=wcs
        )

        # Create FITS HDU
        hdu = fits.PrimaryHDU(cutout.data, header=cutout.wcs.to_header())

        # Filename with RA & Dec
        outname = f"LBG_RA{ra:.5f}_DEC{dec:.5f}.fits"
        outpath = os.path.join(output_dir, outname)

        hdu.writeto(outpath, overwrite=True)
        print(f"Saved: {outname}")

    except Exception as e:
        print(f"Skipping object {i}: {e}")

print(" All snapshots completed.")


  Existing snapshot files deleted.
Saved: LBG_RA357.31920_DEC-31.80416.fits
Saved: LBG_RA357.31957_DEC-31.80350.fits
Saved: LBG_RA357.32054_DEC-31.80367.fits
Saved: LBG_RA357.32340_DEC-31.80240.fits
Saved: LBG_RA357.31761_DEC-31.79918.fits
Saved: LBG_RA357.31917_DEC-31.79131.fits
Saved: LBG_RA356.68488_DEC-31.77336.fits
Saved: LBG_RA356.71953_DEC-31.75385.fits
Saved: LBG_RA356.73744_DEC-31.71398.fits
Saved: LBG_RA356.81101_DEC-31.70875.fits
Saved: LBG_RA357.31202_DEC-31.69604.fits
Saved: LBG_RA357.30792_DEC-31.66562.fits
Saved: LBG_RA357.29601_DEC-31.66358.fits
Saved: LBG_RA357.28298_DEC-31.66305.fits
Saved: LBG_RA357.29223_DEC-31.66273.fits
Saved: LBG_RA357.29060_DEC-31.66269.fits
Saved: LBG_RA357.28958_DEC-31.66295.fits
Saved: LBG_RA357.27311_DEC-31.66259.fits
Saved: LBG_RA357.27863_DEC-31.66072.fits
Saved: LBG_RA357.23990_DEC-31.66055.fits
Saved: LBG_RA357.30271_DEC-31.66088.fits
Saved: LBG_RA357.29281_DEC-31.65990.fits
Saved: LBG_RA357.30135_DEC-31.66113.fits
Saved: LBG_RA357.27192